In [209]:
data = open('input_6.txt',"r").readlines()


In [204]:
data = """....#.....
.........#
..........
..#.......
.......#..
..........
.#..^.....
........#.
#.........
......#..."""

data = data.split()

In [ ]:
data

In [210]:
data = [list(line.strip()) for line in data]
data

[['.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '#',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '#',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '#',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.'],
 ['.',
  '.',
  '.',
  '#',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.'

### Answer 1

In [ ]:
deltas = {
    "^": [(-1,  0),">","|"],
    ">": [( 0,  1),"V","-"],
    "V": [( 1,  0),"<","|"],
    "<": [( 0, -1),"^","-"]
}

grid = [row.copy() for row in data]
start_position = next(
    ((x, line.index('^')) for x, line in enumerate(grid) if '^' in line),
    (None,None)
)
len_x = len(grid)
len_y = len(grid[0])
soldat_here = True
ax,ay = start_position

soldat_here = True
was_turn = False
was_cross  = False
is_cross  = False

while soldat_here:
    dx,dy = deltas[grid[ax][ay]][0]
    nx,ny = (ax + dx, ay + dy)
    if 0 <= nx < len_x and 0 <= ny < len_y:
        if grid[nx][ny] == "#":
            grid[ax][ay] = deltas[grid[ax][ay]][1]
            was_turn = True
        elif grid[nx][ny] in ('.','X','-','|','+') :
            if grid[nx][ny] in ('-','|'):
                is_cross = True
            grid[nx][ny] = grid[ax][ay]
            if was_turn or was_cross:
                grid[ax][ay] = '+'
                was_turn = False
                was_cross = False
            else:
                grid[ax][ay] = deltas[grid[ax][ay]][2]
            was_cross = is_cross
            is_cross = False
            ax,ay = nx,ny
        else:
            print("Wtf?")
    else:
        print("Soldat out!")
        grid[ax][ay] = deltas[grid[ax][ay]][2]
        soldat_here = False


In [ ]:
pd.DataFrame(grid).to_clipboard()

### Answer 2

In [ ]:
def walking_soldat(deltas,
                   grid,
                   obstruct = False,
                   obstruct_coor = (None,None)):
    len_x = len(grid)
    len_y = len(grid[0])
    initial_coor = next(
        ((x, line.index('^')) for x, line in enumerate(grid) if '^' in line),
        (None,None))
    ax,ay = initial_coor

    direction_dict = {}

    if obstruct:
        grid[obstruct_coor[0]][obstruct_coor[1]] = "#"
        if obstruct_coor == initial_coor:
            return True, direction_dict, grid

    soldat_here = True

    while soldat_here:
        current = grid[ax][ay]
        dx,dy = deltas[current][0]
        nx,ny = (ax + dx, ay + dy)

        # Bounds check
        if not(0 <= nx < len_x and 0 <= ny < len_y):
            print("Soldat out!")
            soldat_here = False
            break

        # Wall detection
        if grid[nx][ny] == "#":
            grid[ax][ay] = deltas[current][1]
            continue

        # Loop detection
        if (ax, ay) in direction_dict and current in direction_dict[(ax, ay)]:
            print("🌀 Loop detected at", ax, ay)
            break

        # Track visited directions
        direction_dict.setdefault((ax, ay), []).append(current)

        # Move forward
        grid[nx][ny] = current
        ax, ay = nx, ny

    return soldat_here, direction_dict, grid


In [ ]:

deltas = {
    "^": [(-1,  0),">"],
    ">": [( 0,  1),"V"],
    "V": [( 1,  0),"<"],
    "<": [( 0, -1),"^"]
}

grid = [row.copy() for row in data]

_, direction_dict, _ =  walking_soldat(deltas,grid)

obstruction_coordonates = set()

dict_of_obstacles = {}
for coor, signs in direction_dict.items():
    grid = [row.copy() for row in data]
    is_loop, _, _ = walking_soldat(
        deltas,
        grid,
        obstruct=True,
        obstruct_coor = coor
    )
    if is_loop:
        obstruction_coordonates.add(coor)


len(obstruction_coordonates)

In [218]:

# Chat GPT answer with optimized code
import numpy as np

DIRECTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
DIR_MAP = {'^': 0, '>': 1, 'V': 2, '<': 3}
SYMBOL_MAP = ['^', '>', 'V', '<']

def parse_grid(data):
    grid = np.array([list(row) for row in data])
    start = None
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if grid[i, j] in DIR_MAP:
                start = (i, j)
                dir_idx = DIR_MAP[grid[i, j]]
                grid[i, j] = '.'
                return grid, start, dir_idx
    raise ValueError("No guard found.")

def simulate(grid, start, dir_idx, obstacle=None):
    grid = grid.copy()
    if obstacle:
        ox, oy = obstacle
        if obstacle == start:
            return True  # Illegal placement
        grid[ox, oy] = '#'

    visited = set()
    x, y = start
    while True:
        state = (x, y, dir_idx)
        if state in visited:
            return True  # Loop detected
        visited.add(state)

        dx, dy = DIRECTIONS[dir_idx]
        nx, ny = x + dx, y + dy

        if not (0 <= nx < grid.shape[0] and 0 <= ny < grid.shape[1]):
            return False  # Exited grid

        if grid[nx, ny] == '#':
            dir_idx = (dir_idx + 1) % 4
        else:
            x, y = nx, ny

def find_obstacles(data):
    grid, start, dir_idx = parse_grid(data)
    path = set()

    # Track all positions visited in base run
    x, y = start
    dir_temp = dir_idx
    visited = set()
    while True:
        state = (x, y, dir_temp)
        if state in visited:
            break
        visited.add(state)
        path.add((x, y))
        dx, dy = DIRECTIONS[dir_temp]
        nx, ny = x + dx, y + dy
        if not (0 <= nx < grid.shape[0] and 0 <= ny < grid.shape[1]):
            break
        if grid[nx, ny] == '#':
            dir_temp = (dir_temp + 1) % 4
        else:
            x, y = nx, ny

    # Try placing an obstacle at each visited position
    results = set()
    for pos in path:
        if grid[pos] != '.':
            continue
        if simulate(grid, start, dir_idx, obstacle=pos):
            results.add(pos)
    return results


In [217]:
len(find_obstacles(data))

1576